###Prequisities , run the below code for the setup



In [1]:
!apt-get install -y zstd
!pip install langchain langchain-community langchain-ollama langchain-core chromadb pypdf nest-asyncio --quiet
!pip install wget
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
import time
import wget
import os
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("Downloading llama3 (this may take a minute)...")
!ollama pull llama3
print("Downloading nomic-embed-text...")
!ollama pull nomic-embed-text
print("Ollama setup complete and running!")
PDF_Links=[
    "https://arxiv.org/pdf/1706.03762.pdf",
    "https://arxiv.org/pdf/1810.04805.pdf",
    "https://arxiv.org/pdf/2005.14165.pdf",
    "https://arxiv.org/pdf/2005.11401.pdf",
    "https://arxiv.org/pdf/1908.10084.pdf",
    "https://arxiv.org/pdf/2106.09685.pdf",
    "https://arxiv.org/pdf/2307.09288.pdf"

]
output_directory = "/content/sample_data/content"
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
for link in PDF_Links:
  file_name=link.split('/')[-1]
  wget.download(link, out=output_directory+file_name)
print("The Required files are downloaded")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (14.7 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━

# The main code for building the rag



In [6]:
import ollama
from typing import List
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
def context_load(pdf_directory="/content/sample_data"):
    documents=[]
    for file in os.listdir(pdf_directory): #list the file names of the files in the directory in this case the current folder file names
        if file.endswith('.pdf'):
            pdf_path=os.path.join(pdf_directory,file) # combines nd gives the path as string without the code raissing os related exceptions
            loader = PyPDFLoader(pdf_path) #retriving the contents from the pdf
            pages = loader.load() #stores eaches content
            clean_source_name = file.replace(".pdf", "").title() # cleaner pdf name without .pdf
            for page in pages:
                page.metadata["source_paper"] = clean_source_name
                page.metadata["page_number"] = page.metadata.get("page", 0) + 1 #ensure the first page number key is 1
                documents.append(page)
    print(f"succsfully retrived contents and saved the contents of {len(documents)}")
    return documents
pdf_directory='enter your own directory'
#n 2nd step is we divide the got data to be fed that is we divide the data into small chunks
def split_documents(documents):
    splitter=RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=50,
        separators=['\n','\n\n',' ']
    )
    chunks=splitter.split_documents(documents)
    print('the chunksn of the document has been created')
    print(f"the number of chunks formed is {len(chunks)}")
    return chunks
#the third step is that we save the chunks of the data in a vector data base by coverting it into a vector
def vector_store_creation(chunks, database_directory="./chroma_db"):
    embeddings=OllamaEmbeddings(model='nomic-embed-text') #loading the embedding model
    vector_store=Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=database_directory
    )                                       #embedding all the chunks with the embedding model specified above
    return vector_store
prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks.
    Use the following documents to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences maximum and keep the answer concise:
    Question: {question}
    Documents: {documents}
    Answer:
    """,
    input_variables=["question", "documents"] #making a prompt template
)
llm=ChatOllama(
    model='llama3',
    temperature=0
) #loading an local llm llama3
documents=context_load()
chunks=split_documents(documents)
vector_store=vector_store_creation(chunks)
retriever = vector_store.as_retriever(k=4)
rag_chain=prompt | llm | StrOutputParser()
class RAG_Application:
    def __init__(self,retriver,rag_chain):
        self.retriver=retriver
        self.rag_chain=rag_chain
    def run(self,question):
        documents=self.retriver.invoke(question) #fetch the related documemts
        doc_texts="\\n".join([doc.page_content for doc in documents])
        answer=self.rag_chain.invoke({"question":question,"documents":documents}) #passing the query throught the ragb pipeline and rertriving
        return answer
rag_application=RAG_Application(retriever,rag_chain)



succsfully retrived contents and saved the contents of 239
the chunksn of the document has been created
the number of chunks formed is 953


Few sample questions without chat bot

In [7]:
question="How does self-attention differ from recurrence?"
answer=rag_application.run(question)
print(answer)

Self-attention differs from recurrence in that it connects all positions with a constant number of sequentially executed operations, whereas a recurrent layer requires O(n) sequential operations. This means that self-attention is faster than recurrence when processing sequences of varying lengths.


In [ ]:
question='What problem does RAG solve?'
answer=rag_application.run(question)
print(answer)

RAG (Reinforced Generation) solves the problem of generating responses that are more factual and specific than those produced by state-of-the-art generation models like BART. It achieves this by learning to retrieve relevant information from a large corpus of text, which allows it to generate responses that combine content from multiple documents.


In [8]:
from google.colab import output
import IPython
def ask_chatbot(question):
    try:
        answer = rag_application.run(question)
        return answer
    except Exception as e:
        return f"Error processing request: {str(e)}"
output.register_callback('ask_chatbot', ask_chatbot)

# Building Chatbot using HTML

In [9]:
css_content='''#chat-container {
    width: 100%;
    max-width: 800px;
    margin: 0 auto;
    border: 1px solid #ddd;
    border-radius: 8px;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
    box-shadow: 0 4px 8px rgba(0,0,0,0.1);
  }
  #chat-history {
    height: 400px;
    overflow-y: auto;
    padding: 15px;
    background: #f8f9fa;
    display: flex;
    flex-direction: column;
    gap: 10px;
  }
  .message {
    padding: 10px 15px;
    border-radius: 18px;
    max-width: 75%;
    word-wrap: break-word;
    line-height: 1.4;
  }
  .user-msg {
    background: #007bff;
    color: white;
    align-self: flex-end;
    border-bottom-right-radius: 4px;
  }
  .bot-msg {
    background: #e9ecef;
    color: #212529;
    align-self: flex-start;
    border-bottom-left-radius: 4px;
  }
  #input-container {
    display: flex;
    padding: 15px;
    border-top: 1px solid #ddd;
    background: white;
    border-bottom-left-radius: 8px;
    border-bottom-right-radius: 8px;
  }
  #chat-input {
    flex-grow: 1;
    padding: 10px 15px;
    border: 1px solid #ccc;
    border-radius: 20px;
    outline: none;
    font-size: 16px;
  }
  #chat-input:focus {
    border-color: #007bff;
  }
  #send-btn {
    margin-left: 10px;
    padding: 10px 20px;
    background: #28a745;
    color: white;
    border: none;
    border-radius: 20px;
    cursor: pointer;
    font-size: 16px;
    font-weight: bold;
    transition: background 0.2s;
  }
  #send-btn:hover {
    background: #218838;
  }
  .typing {
    font-style: italic;
    color: #6c757d;
  }
'''
html_content='''<div id="chat-container">
  <div id="chat-history">
    <div class="message bot-msg">Hello! I am your RAG assistant. Ask me anything based on the loaded documents.</div>
  </div>
  <div id="input-container">
    <input type="text" id="chat-input" placeholder="Type your question here..." onkeypress="handleKeyPress(event)">
    <button id="send-btn" onclick="sendMessage()">Send</button>
  </div>
</div>'''
js_content='''async function sendMessage() {
    const inputField = document.getElementById('chat-input');
    const history = document.getElementById('chat-history');
    const question = inputField.value.trim();

    if (!question) return;

    //  Add user message to UI
    history.innerHTML += `<div class="message user-msg">${question}</div>`;
    inputField.value = ''; // clear input
    history.scrollTop = history.scrollHeight; // auto-scroll to bottom

    //  Add "Thinking..." placeholder
    const thinkingId = 'msg-' + Date.now();
    history.innerHTML += `<div class="message bot-msg typing" id="${thinkingId}">Thinking...</div>`;
    history.scrollTop = history.scrollHeight;

    try {
      //  Call the Python function we registered via Colab Kernel
      const result = await google.colab.kernel.invokeFunction('ask_chatbot', [question], {});

      // Extract the text content from the Python return object
      let answer = result.data['text/plain'];

      // Clean up Python string output formatting (strip surrounding quotes and fix linebreaks)
      answer = answer.replace(/^'|'$/g, '').replace(/^"|"$/g, '').replace(/\\\\n/g, '<br>');

      //  Update the thinking placeholder with the real answer
      const responseEl = document.getElementById(thinkingId);
      responseEl.classList.remove('typing');
      responseEl.innerHTML = answer;
    } catch (error) {
      document.getElementById(thinkingId).innerText = "An error occurred while fetching the answer.";
      console.error(error);
    }

    history.scrollTop = history.scrollHeight;
  }

  // Allow pressing Enter to send
  function handleKeyPress(event) {
    if (event.key === 'Enter') {
      sendMessage();
    }
  }'''
final = f"""
<style>
{css_content}
</style>

{html_content}

<script>
{js_content}
</script>
"""
IPython.display.display(IPython.display.HTML(final))